# 알고리즘 기말 프로젝트 — Score Function Design

- **제출일**: `<YYYY.MM.DD HH:MM>`
- **파일명**: `이름_학번_화학제품군.ipynb`  (예: 신현길_12312312_pesticide.ipynb)

## 학번 / 이름

- **학번**: ` 20251259 `
- **이름**: ` 신헤지   `
- **score에 대한 간략한 설명**: `fragrance`-likeness를 계산하는 score (cosmetic / food additive / fragrance / ... )

## 채점 기준 (100점)

| 영역 | 배점 | 만점 기준 |
|---|---|---|
| **(1) Negative 데이터 준비** | 20 | "구조 유사도"를 통해 negative 집합을 찾아낸 기준? (유사도 측정 방법 & Structural similarity 기준 설정) |
| **(2) Score 함수 설계** | 20 | **(a) 분자 속성 범위 (전체 데이터에서 property 범위 계산 방식)** + **(b) alert 구조 패턴(scaffold 및 부분 구조의 smarts 패턴)** 두 요소 모두 포함. score에 대한 설명은 markdown에 기재. |
| **(3) Score 평가 — goodness** | 20 | positive vs negative 점수 분포 비교 (히스토그램/ROC/AUC 등 score 성능의 근거가 되는 시각화 자료 제시) |
| **(4) 설명** | 10 | 각 알고리즘을 mermaid를 이용해서 표현하고 설명글 추가 (markdown 및 주석으로 표기) |


### 가산점 (선택)

| 가산 | 점수 | 조건 |
|---|---|---|
| **(A) 다른 화학 제품군 score** | +10 | pesticide 외 1종 이상(cosmetic / food additive / fragrance / surfactant / dye 등)의 PubChem classification 데이터로 별도 score 함수 설계 + 평가 |
| **(B) Score 기반 구조 생성** | +10 | 본인 score 를 reward 로 사용해 score가 개선된 새로운 구조 생성. |
| **(C) 계산 자원과 계산 시간** | +10 | mpi를 이용해서 대량의 자원으로 계산 시간을 대폭 줄이거나, local 환경에서 합리적으로 계산이 진행될 수 있도록 문제를 효율적으로 압축시킨 방법 적용 (mpi script와 계산 결과에 대한 log 필요) |

### 제출 결과물 (결과를 재현하기 위해 필요한 파일들)
1. ipynb (mpi를 사용했다면, mpi4py script)
2. data files (pesticide, cosmetics, food additives, drug, ..., format: csv)
3. negative list file (format: csv)
4. score 평가 시각화 자료 (mpi에서 실행해서 얻은 plot은 notebook markdown에 삽입)

---
# Task 1. Negative 데이터 준비 (25점)

**문제**: 양성(positive) 분자와 "구조적으로 다른" 분자 집합을 어떻게 만들 것인가?

Score 함수의 평가는 **양성과 음성을 얼마나 잘 구분하는가** 로 측정합니다. 그러려면 먼저 음성 집합을 정의해야 합니다.

**📝 본인 선택과 이유 (직접 작성):** 

- 선택한 기준: drug dataset을 nagative dataset으로 사용하였다. 
- 이유: drug molecule은 fragrance molecule과 비교했을 때 사용 목적과 구조적 특성이 다르기 때문이다. 
fragrance molecule은 일반적으로 비교적 작은 분자량과 낮은 극성을 가지고 있고 
drug molecule은 생체 활성 및 약물 전달을 위해 보다 복잡한 구조가 많다. (다양한 작용기를 포함함.)
따라서 fragrance-likeness score를 평가하기 위한 neagative dataset으로 선택하였다. 

In [4]:
# 셀은 필요한 만큼 추가. (mpi로 만들어낸 결과는 markdown에 추가. python script는 별도 제출.)

In [1]:
import pandas as pd 

fragrance_df = pd.read_csv("fragrance.csv")
drug_df = pd.read_csv("drug.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'drug.csv'

In [ ]:
fragrance_df.head()

In [ ]:
drug_df.head()

In [ ]:
# RDkit 용 valid molecule 만들기 

from rdkit import Chem 

fragrance_valid = []
drug_valid = []

for s in fragrance_df["SMIILES"].dropna():

    if Chem.MolFromSmiles(s):
        fragrance_valid.append(s)

for s in drug_df["SMILES"].dropna():

    if Chem.MolFromSmiles(s):
        drug_valid.append(s)

print(len(fragrance_valid))
print(len(drug_valid))

In [ ]:
# descriptor 계산 
# fragrance와 drug 의 화학적 특징 계산 

from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescrptors 

def make_desc(smiles_list):

    mols = [Chem.MolFromSmiles(s) for s in smiles_list]

    return pd.DataFrame({
        "MW": [Descriptors.MolWt(m) for m in mols], 
        "logP": [Descriptors.MolLogP(m) for m in mols], 
        "TPSA": [rdMolDescriptors.CalcTPSA(m) for m in mols], 
        "HBA": [rdMolDescriptors.CalcNumHBA(m) for m in mols], 
        "HBD": [rdMolDescriptors.CalcNumHBD(m) for m in mols]
    })

fragrance_desc = make_desc(fragrance_valid)
drug_desc = make_desc(drug_valid)

In [ ]:
# histogram 그려보기 - fragrance와 drug 의 차이 시각화 

import matplotlib.pyplot as plt 

for col in fragrance_desc.columns:
    plt.figure(figsize = (5, 3))
    
    plt.hist(fragrance_desc[col], bins =30, alpha = 0.7, label="Fragrance")
    plt.hist(drug_desc[col], bins = 30, alpha = 0.7, label = "Drug")

    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"{col} Distribution")

    plt.legend()

    plt.show()

SyntaxError: invalid syntax (47306076.py, line 3)

Fragrance dataset은 MW가 주로 100~300 범위에 집중되었다.
반면 drug dataset은 보다 넓은 분포를 나타냈다.

또한 fragrance molecule은 상대적으로 낮은 TPSA 값을 가지는 경우가 많았으며,
이는 향료 분자가 비교적 낮은 극성을 가진다는 특성과 관련이 있다고 해석하였다.

이러한 descriptor 차이를 기반으로 fragrance-likeness score를 설계하였다.

**📝 Task 1. 결과 해석 (아래는 예시):**

- 두 분포가 어디서 갈라지는가? (예: 음성 봉우리 ~0.15, 양성 봉우리 ~0.35)
- 어떤 실험을 거쳐서 구조 유사도 기준 값을 설정했는가?

---
# Task 2. Score 함수 설계 (35점)

**문제**: "positive-likeness" 점수를 계산해주는 함수 개발.

**Scoring 방식** 
1. **(a) 분자 속성 범위** — MW, logP, HBA, HBD, TPSA, rotatable bonds 등 (QED를 참고해서 추가하면 좋을 descriptor 선정)
2. **(b) SMARTS 패턴** — 양성에서 자주 나타나는 작용기/하부구조를 포함 or 양성에서 나타나지 않는 구조패턴을 찾아서 제외.

두 점수를 어떻게 결합할지(합/곱/가중합/기하평균/...)도 직접 결정.

In [3]:
# 셀은 필요한 만큼 추가.

**📝 Score 함수 설계 근거 (아래는 예시):**

- 어떤 SMARTS 패턴을 골랐고 그 이유는?
- 결합 방식과 가중치 선택의 이유는?

---
# Task 3. Score 평가 — Goodness of the score (30점)

**문제**: 본인이 만든 score 함수가 양성과 음성을 얼마나 잘 구분하는가?
- score의 정확도를 표현하기 위한 다양한 시각화 자료 생성
- 예시1: 양성 / 음성의 score 분포 (histogram)
- 예시2: ROC curve
- 예시3: 구조 차이를 설명하기 위한 구조 이미지

In [ ]:
# 시각화

**📝 Score 평가 해석 (아래는 예시):**
1. score가 좋다면, 어떤식의 결과가 예상될까? (예상을 확인할 수 있는 시각화 방법?)
2. 농약에만 있는 구조? 농약에는 없지만 의약품에서만 나타나는 구조? (비교군이 의약품인 경우)

---
# (가산 A) 다른 화학 제품군 score 함수 (+10)

Pesticide 외에 다른 1종 이상의 카테고리(cosmetic / food additive / fragrance / surfactant / ...)에 대해 같은 절차로 score 함수를 만들고 평가 (별도 ipynb로 제출)

PubChem [Classification Browser](https://pubchem.ncbi.nlm.nih.gov/classification) 에서 원하는 카테고리의 CSV 를 다운로드

**시도 하지 않은 경우 아래 부분은 빈칸으로 제출.**

---
# (가산 B) Score 기반 구조 생성 (+10)

본인 score 를 reward 로 사용해 새 구조를 생성. 알고리즘은 자유 (greedy / DP / random walk / 본인 방법). mpi를 사용해서 대규모로 구조를 생성했다면, 생성된 구조의 분포 시각화 (예: histogram, scatter plot 등)을 

**시도하지 않은 경우 아래 부분은 빈칸으로 제출.**

---
# (가산 C) 계산 자원과 계산 시간 (+10)

mpi 혹은 알고리즘 효율화를 통해 계산 시간이 단축되었음을 설명한 경우 주어지는 가산점.
코드에 소요 시간 계산을 위한 코드를 삽입. mpi를 사용해서 시간이 단축되었음을 설명 혹은 문제를 합리적으로 단순화시켜서 계산시간을 단축시켰음을 설명. (계산 시간 비교를 통해 알고리즘의 효율성을 설명해야 함.)